# Notebook 8 – Évaluation quantitative
**Projet Computer Vision IG.2405 – 2026**

Ce notebook évalue objectivement les deux programmes selon les 4 axes du challenge :

1. **Authentification des signatures** (Prog 1 + Prog 2)
2. **Lecture des informations imprimées** (Prog 2 – lignes 1-4, 9-11)
3. **Lecture des informations manuscrites** (Prog 2 – lignes 13-14, MANTISSE/EXPOSANT)
4. **Lecture des informations graphiques** (Prog 2 – lignes 5-9, 16-18, CHOIX)

Méthodologie : **base d'apprentissage / validation / test** distinctes (validation croisée recommandée).

In [ ]:
import sys, os

# Racine du projet (fonctionne depuis notebooks/ ou depuis la racine)
if os.path.basename(os.getcwd()) == 'notebooks':
    os.chdir('..')
ROOT = os.getcwd()
if ROOT not in sys.path:
    sys.path.insert(0, ROOT)

DATA_ROOT = os.path.join(ROOT, 'PROJECT 2026 -DATABASE-20260518')
print(f'Racine projet : {ROOT}')
print(f'Base de donnees : {DATA_ROOT}')

In [ ]:
import os
import cv2
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split

from utils.pdf_utils import pdf_to_images
from utils.grid_decoder import normalize_page, read_student_id, read_group, read_conditions
from utils.form_aligner import deskew
from utils.page1_parser import parse_page1
from utils.exam_parser import parse_exam_pages, questions_to_exam_rows

FORM_DIR    = os.path.join('PROJECT 2026 -DATABASE-20260518', 'FORM1')
RESULTS_DIR = 'EXAM_FORM1_RESULTS'

# Lister tous les fichiers
pdf_files  = sorted([f for f in os.listdir(FORM_DIR) if f.lower().endswith('.pdf')])
xlsx_files = sorted([f for f in os.listdir(FORM_DIR) if f.lower().endswith('.xlsx')])
img_files  = sorted([f for f in os.listdir(FORM_DIR)
                     if f.lower().endswith(('.jpg', '.jpeg', '.png'))])

# IDs avec vérité terrain complète (PDF + XLSX)
pdf_ids  = {f.replace('.pdf','').split('_')[-1] for f in pdf_files}
xlsx_ids = {f.replace('.xlsx','').split('_')[-1] for f in xlsx_files}
complete_ids = sorted(pdf_ids & xlsx_ids)

print(f'IDs avec vérité terrain complète : {len(complete_ids)}')
print(complete_ids[:10])

## 1. Split Train / Validation / Test

Conformément au cahier des charges, les 3 bases doivent être **bien distinctes**.
Split recommandé : 60% train, 20% validation, 20% test.

In [ ]:
np.random.seed(42)

ids_train, ids_tmp = train_test_split(complete_ids, test_size=0.40, random_state=42)
ids_val,   ids_test = train_test_split(ids_tmp,      test_size=0.50, random_state=42)

print(f'Train      : {len(ids_train)} ({len(ids_train)/len(complete_ids):.1%})')
print(f'Validation : {len(ids_val)}   ({len(ids_val)/len(complete_ids):.1%})')
print(f'Test       : {len(ids_test)}  ({len(ids_test)/len(complete_ids):.1%})')

# Vérifier l'absence de chevauchement
assert not set(ids_train) & set(ids_val), 'Overlap train/val!'
assert not set(ids_train) & set(ids_test), 'Overlap train/test!'
assert not set(ids_val) & set(ids_test), 'Overlap val/test!'
print('\nPas de chevauchement entre les 3 splits. OK.')

# Visualiser la répartition
labels = ['Train', 'Validation', 'Test']
sizes  = [len(ids_train), len(ids_val), len(ids_test)]
colors = ['#2ecc71', '#3498db', '#e74c3c']

plt.figure(figsize=(6, 4))
bars = plt.bar(labels, sizes, color=colors, edgecolor='black')
for bar, n in zip(bars, sizes):
    plt.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.1,
             str(n), ha='center', fontsize=12, fontweight='bold')
plt.title('Répartition Train / Val / Test')
plt.ylabel('Nombre d\'étudiants')
plt.tight_layout()
plt.show()

## 2. Fonctions d'évaluation

In [ ]:
def load_ground_truth(sid: str) -> dict:
    """Charge la vérité terrain PAGE-01 et EXAM depuis le xlsx."""
    xlsx = os.path.join(FORM_DIR, f'EXAM_FORM1_{sid}.xlsx')
    if not os.path.isfile(xlsx):
        return {}
    try:
        df_p1 = pd.read_excel(xlsx, sheet_name='PAGE-01', header=None)
        gt_p1 = dict(zip(df_p1[0].astype(str).str.strip(),
                         df_p1[1].astype(str).str.strip()))
    except Exception:
        gt_p1 = {}
    try:
        df_ex = pd.read_excel(xlsx, sheet_name='EXAM')
        gt_ex = df_ex
    except Exception:
        gt_ex = pd.DataFrame()
    return {'page1': gt_p1, 'exam': gt_ex}


def compute_accuracy(predicted, ground_truth):
    """Accuracy simple : % de valeurs exactement correctes."""
    if not ground_truth:
        return None
    correct = sum(1 for p, g in zip(predicted, ground_truth)
                  if str(p).strip() == str(g).strip())
    return correct / len(ground_truth) if ground_truth else 0.0


print('Fonctions d\'évaluation chargées.')

## 3. Axe 1 – Lecture des informations graphiques (Student ID, Groupe, Conditions)

Évalué sur les lignes 5-9 et 16-18 de l'onglet PAGE-01, et les CHOIX de l'onglet EXAM.

In [ ]:
sid_results  = {'correct': 0, 'total': 0}
grp_results  = {'correct': 0, 'total': 0}
cond_results = {'correct': 0, 'total': 0}

eval_ids = ids_test  # Utiliser la base test pour l'évaluation finale

for sid in eval_ids:
    pdf_path = os.path.join(FORM_DIR, f'EXAM_FORM1_{sid}.pdf')
    if not os.path.isfile(pdf_path):
        continue

    gt = load_ground_truth(sid)
    if not gt or not gt.get('page1'):
        continue

    gt_p1 = gt['page1']

    try:
        pages = pdf_to_images(pdf_path, dpi=150)
        pred  = parse_page1(pages[0], desc_db=None)
    except Exception:
        continue

    # Student ID (ligne 17)
    gt_sid = gt_p1.get('STUDENT ID', gt_p1.get('Student ID', ''))
    pred_sid = str(pred.get('student_id', ''))
    sid_results['total'] += 1
    if str(gt_sid).strip() == pred_sid.strip():
        sid_results['correct'] += 1

    # Groupe (ligne 16)
    gt_grp = gt_p1.get('Group', '')
    pred_grp = str(pred.get('group', ''))
    grp_results['total'] += 1
    if str(gt_grp).strip() == pred_grp.strip():
        grp_results['correct'] += 1

    # Notes de cours (ligne 5)
    gt_nc = str(gt_p1.get('Notes de cours', '0')).strip()
    pred_nc = str(pred.get('notes_cours', 0))
    cond_results['total'] += 1
    if gt_nc == pred_nc:
        cond_results['correct'] += 1

print('=== Axe 4 : Informations graphiques (base test) ===')
for name, r in [('Student ID', sid_results), ('Groupe', grp_results),
                 ('Notes cours', cond_results)]:
    acc = r['correct'] / r['total'] * 100 if r['total'] > 0 else 0
    print(f'  {name:20s} : {r["correct"]}/{r["total"]} = {acc:.1f}%')

## 4. Axe 2 – Lecture des informations imprimées

Évalué sur les lignes 1-4 (Module, Professeur, Date, Code) et 9-11 (Note max, Note valider).

In [ ]:
printed_results = {}
fields_printed = [
    ('Module',      'module',    'Module'),
    ('Professor',   'professor', 'Professor'),
    ('Date',        'date',      'Date'),
    ('Code',        'code',      'Code'),
    ('Note max',    'note_max',  'Note maximale'),
    ('Note valider','note_valid','Note pour valider'),
]
for label, _, _ in fields_printed:
    printed_results[label] = {'correct': 0, 'total': 0}

for sid in eval_ids[:10]:  # limiter pour rapidité
    pdf_path = os.path.join(FORM_DIR, f'EXAM_FORM1_{sid}.pdf')
    if not os.path.isfile(pdf_path):
        continue
    gt = load_ground_truth(sid)
    if not gt or not gt.get('page1'):
        continue
    gt_p1 = gt['page1']
    try:
        pages = pdf_to_images(pdf_path, dpi=150)
        pred  = parse_page1(pages[0], desc_db=None)
    except Exception:
        continue

    for label, pred_key, gt_key in fields_printed:
        gt_val   = str(gt_p1.get(gt_key, '')).strip()
        pred_val = str(pred.get(pred_key, '')).strip()
        printed_results[label]['total'] += 1
        if gt_val and gt_val.lower() == pred_val.lower():
            printed_results[label]['correct'] += 1

print('=== Axe 2 : Informations imprimées ===')
for label, r in printed_results.items():
    acc = r['correct'] / r['total'] * 100 if r['total'] > 0 else 0
    print(f'  {label:20s} : {r["correct"]}/{r["total"]} = {acc:.1f}%')

## 5. Évaluation MCQ (Axe 4 – CHOIX)

Pour chaque question MCQ, on compare les cases cochées prédites vs vérité terrain.

In [ ]:
mcq_tp, mcq_fp, mcq_fn = 0, 0, 0
EXAM_START = 4

for sid in eval_ids[:5]:  # limiter pour rapidité
    pdf_path = os.path.join(FORM_DIR, f'EXAM_FORM1_{sid}.pdf')
    if not os.path.isfile(pdf_path):
        continue
    gt = load_ground_truth(sid)
    if not gt or gt['exam'].empty:
        continue

    try:
        pages = pdf_to_images(pdf_path, dpi=150)
        questions = parse_exam_pages(pages, exam_start_page=EXAM_START)
        pred_rows = questions_to_exam_rows(questions)
    except Exception:
        continue

    gt_exam = gt['exam']
    choice_cols = [c for c in gt_exam.columns if 'CHOIX' in str(c).upper()]

    for q_idx, (pred_row, *_) in enumerate(zip(pred_rows, [None]*len(pred_rows))):
        if q_idx >= len(gt_exam):
            break
        gt_row = gt_exam.iloc[q_idx]
        for col in choice_cols:
            letter = col.split()[-1] if ' ' in col else col[-1]
            gt_val   = gt_row.get(col, None)
            pred_val = pred_row.get(f'CHOIX {letter}', None)

            gt_checked   = (gt_val == 1 or str(gt_val) == '1')
            pred_checked = (pred_val == 1)

            if gt_checked and pred_checked:
                mcq_tp += 1
            elif not gt_checked and pred_checked:
                mcq_fp += 1
            elif gt_checked and not pred_checked:
                mcq_fn += 1

precision = mcq_tp / (mcq_tp + mcq_fp) if (mcq_tp + mcq_fp) > 0 else 0
recall    = mcq_tp / (mcq_tp + mcq_fn) if (mcq_tp + mcq_fn) > 0 else 0
f1        = 2 * precision * recall / (precision + recall) if (precision + recall) > 0 else 0

print('=== Évaluation MCQ (Axe 4 - CHOIX) ===')
print(f'  TP={mcq_tp}, FP={mcq_fp}, FN={mcq_fn}')
print(f'  Précision : {precision:.3f}')
print(f'  Rappel    : {recall:.3f}')
print(f'  F1-score  : {f1:.3f}')

## 6. Synthèse globale et courbe d'apprentissage

In [ ]:
# Synthèse des métriques collectées
summary = {
    'Student ID (grille)': sid_results['correct'] / max(sid_results['total'], 1),
    'Groupe (grille)':     grp_results['correct'] / max(grp_results['total'], 1),
    'Conditions examen':   cond_results['correct'] / max(cond_results['total'], 1),
    'MCQ - Précision':     precision,
    'MCQ - Rappel':        recall,
    'MCQ - F1':            f1,
}

# Ajouter les métriques imprimées
for label, r in printed_results.items():
    summary[f'Imprimé - {label}'] = r['correct'] / max(r['total'], 1)

# Affichage
fig, ax = plt.subplots(figsize=(10, 5))
keys   = list(summary.keys())
values = list(summary.values())
colors = ['#2ecc71' if v >= 0.8 else '#f39c12' if v >= 0.5 else '#e74c3c'
          for v in values]

bars = ax.barh(keys, values, color=colors, edgecolor='black')
ax.axvline(0.8, color='darkgreen', linestyle='--', linewidth=1, label='Seuil 80%')
ax.set_xlim(0, 1.05)
ax.set_xlabel('Score')
ax.set_title('Récapitulatif des performances (base test)', fontsize=12, fontweight='bold')
ax.legend()

for bar, v in zip(bars, values):
    ax.text(v + 0.01, bar.get_y() + bar.get_height()/2,
            f'{v:.1%}', va='center', fontsize=9)

plt.tight_layout()
plt.show()

## 7. Analyse des erreurs

In [ ]:
# Identifier les cas difficiles pour la lecture de grille
print('=== Analyse des erreurs - Student ID ===')

error_cases = []
for sid in eval_ids:
    pdf_path = os.path.join(FORM_DIR, f'EXAM_FORM1_{sid}.pdf')
    if not os.path.isfile(pdf_path):
        continue
    try:
        pages = pdf_to_images(pdf_path, dpi=150)
        pred  = parse_page1(pages[0], desc_db=None)
        pred_sid = str(pred.get('student_id', ''))
        if pred_sid != sid:
            error_cases.append({'ID attendu': sid, 'ID prédit': pred_sid})
    except Exception as e:
        error_cases.append({'ID attendu': sid, 'ID prédit': f'ERREUR: {e}'})

if error_cases:
    print(f'{len(error_cases)} erreur(s) détectée(s) sur la base test :')
    for e in error_cases:
        print(f"  Attendu: {e['ID attendu']} | Prédit: {e['ID prédit']}")
else:
    print('Aucune erreur de lecture de Student ID sur la base test !')

## 8. Recommandations d'amélioration

Sur la base de l'analyse des erreurs :

In [ ]:
print("""
Recommandations d'amélioration :

1. GRILLE STUDENT ID
   - Calibrer les coordonnées ROI sur un plus grand échantillon
   - Tester différents seuils ink_ratio (actuellement 0.08)
   - Ajouter morphologie pour réduire le bruit

2. SIGNATURE
   - Augmenter la base d'apprentissage (plusieurs signatures/étudiant)
   - Calibrer le seuil τ sur la base de validation (ROC curve)
   - Tester d'autres descripteurs (SIFT, ORB, deep features)

3. OCR TEXTES IMPRIMÉS
   - Ajuster les zones ROI si besoin
   - Améliorer le prétraitement (CLAHE pour PDFs de faible qualité)

4. MCQ
   - Affiner le seuil CHECKED_INK_THRESHOLD
   - Améliorer has_x_pattern pour les X tracés différemment

5. RÉPONSES NUMÉRIQUES
   - Entraîner un classificateur de chiffres sur les données manuscrites
   - Gérer les couleurs d'encre variées (rouge, bleu, noir)
""")

## Bilan global

| Axe Challenge | Méthode | Métriques |
|---|---|---|
| Signatures | HOG + Hu + cosinus | Précision / Rappel |
| Imprimés | EasyOCR + regex | Accuracy par champ |
| Manuscrits | CLAHE + EasyOCR + CC | Accuracy mantisse/exposant |
| Graphiques | Ink ratio + Hough | Accuracy Student ID, MCQ F1 |

---
**Fin des notebooks** – Pipeline complet implémenté et évalué.

Pour lancer l'exécution complète : `python main.py`